In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: calcular promedios trimestrales de material particulado con base en los criterios estipulados en la NOM-025-SSA1-2014. 
# Periodo: 2000-2019
#Se puede usar para PM10 y PM2.5 solo se debe modificar la "Configuración"
# ==========================================

In [16]:
import pandas as pd
import numpy as np
import os

# ========== CONFIGURACIÓN ==========
ruta_archivo = "AGREGAR RUTA DEL ARCHIVO"
ruta_salida = "AGREGAR RUTA DE CARPETA DE SALIDA"
nombre_archivo_salida = "AGREGAR NOMBRE DE SALIDA.csv”

# ========== CARGAR DATOS ==========
df = pd.read_csv(ruta_archivo, parse_dates=["FECHA"])

# ========== PREPARAR DATOS ==========
df["AÑO"] = df["FECHA"].dt.year
df["TRIMESTRE"] = df["FECHA"].dt.month.map({
    1: 1, 2: 1, 3: 1,
    4: 2, 5: 2, 6: 2,
    7: 3, 8: 3, 9: 3,
    10: 4, 11: 4, 12: 4
})

# Identificar columnas de municipios (todas menos auxiliares)
columnas_municipios = df.columns.difference(["FECHA", "AÑO", "TRIMESTRE"])

# ========== REDONDEO A ENTERO PERSONALIZADO ==========
def redondeo_entero(valor):
    if pd.isna(valor):
        return np.nan
    primer_decimal = int((valor * 10) % 10)
    return np.floor(valor) if primer_decimal <= 4 else np.ceil(valor)

# ========== CALCULAR PROMEDIOS TRIMESTRALES ==========
resultados = []

for (año, trimestre), grupo in df.groupby(["AÑO", "TRIMESTRE"]):
    dias_totales = grupo["FECHA"].nunique()
    fila_resultado = {"AÑO": año, "TRIMESTRE": trimestre}
    
    for col in columnas_municipios:
        datos_validos = pd.to_numeric(grupo[col], errors='coerce').dropna()
        if len(datos_validos) >= 0.75 * dias_totales:
            promedio = datos_validos.sum() / len(datos_validos)
            fila_resultado[col] = int(redondeo_entero(promedio))
        else:
            fila_resultado[col] = np.nan
    
    resultados.append(fila_resultado)

# ========== GUARDAR RESULTADOS ==========
df_resultado = pd.DataFrame(resultados)
ruta_completa = os.path.join(ruta_salida, nombre_archivo_salida)
df_resultado.to_csv(ruta_completa, index=False, encoding="utf-8", na_rep="NaN")

print("Promedio trimestral calculado exitosamente en:", ruta_completa)

Promedio trimestral calculado exitosamente en: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MATERIAL PARTICULADO/PM2.5/4.PROM_TRIMESTRAL_PM2.5.csv
